## Cell 1 — Install Dependencies

In [ ]:
!pip install -q huggingface-hub llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 \
    --no-cache-dir
!pip install -q pyngrok nest_asyncio sentence-transformers torch InstructorEmbedding

## Cell 2 — Load Kaggle Secrets

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ['NGROK_AUTHTOKEN'] = user_secrets.get_secret('NGROK_AUTHTOKEN')

## Cell 3 — Full Server Code

In [ ]:
import asyncio
import base64
import threading
import time
import json
from typing import Optional, List, Dict, Any, Union
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from llama_cpp import Llama
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
from InstructorEmbedding import INSTRUCTOR
import uvicorn
import logging
import requests
import nest_asyncio
from pyngrok import ngrok
import atexit
from datetime import datetime, timedelta
import uuid
import numpy as np

nest_asyncio.apply()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

LLM_MODEL_REPO  = 'ggml-org/gpt-oss-20b-GGUF'
LLM_MODEL_FILE  = 'gpt-oss-20b-mxfp4.gguf'
LLM_MODEL_ID    = 'gpt-oss-20b'

EMBEDDING_MODEL_E5_ID  = 'intfloat/multilingual-e5-large'
EMBEDDING_MODEL_E5_DIM = 1024

EMBEDDING_MODEL_INSTRUCTOR_ID  = 'hkunlp/instructor-large'
EMBEDDING_MODEL_INSTRUCTOR_DIM = 768

DEFAULT_EMBEDDING_MODEL_ID = EMBEDDING_MODEL_E5_ID

# ── Canonical model-id aliases (normalise short names → full HF path) ──────────
MODEL_ALIASES: Dict[str, str] = {
    'multilingual-e5-large': EMBEDDING_MODEL_E5_ID,
    'e5-large':               EMBEDDING_MODEL_E5_ID,
    'e5':                     EMBEDDING_MODEL_E5_ID,
    EMBEDDING_MODEL_E5_ID:    EMBEDDING_MODEL_E5_ID,
    'instructor-large':       EMBEDDING_MODEL_INSTRUCTOR_ID,
    'instructor':             EMBEDDING_MODEL_INSTRUCTOR_ID,
    EMBEDDING_MODEL_INSTRUCTOR_ID: EMBEDDING_MODEL_INSTRUCTOR_ID,
    '':                       EMBEDDING_MODEL_E5_ID,
}


class Message(BaseModel):
    role:    str = Field(..., description='Role: system, user, or assistant')
    content: str = Field(..., description='Message content')


class ChatCompletionRequest(BaseModel):
    model:             str                            = Field(default=LLM_MODEL_ID)
    messages:          List[Message]                  = Field(...)
    temperature:       Optional[float]                = Field(default=0.7,  ge=0.0, le=2.0)
    max_tokens:        Optional[int]                  = Field(default=500,  ge=1)
    stream:            Optional[bool]                 = Field(default=False)
    top_p:             Optional[float]                = Field(default=1.0,  ge=0.0, le=1.0)
    frequency_penalty: Optional[float]                = Field(default=0.0,  ge=-2.0, le=2.0)
    presence_penalty:  Optional[float]                = Field(default=0.0,  ge=-2.0, le=2.0)
    stop:              Optional[Union[str, List[str]]] = Field(default=None)


class CompletionRequest(BaseModel):
    model:       str                            = Field(default=LLM_MODEL_ID)
    prompt:      str                            = Field(...)
    temperature: Optional[float]                = Field(default=0.7,  ge=0.0, le=2.0)
    max_tokens:  Optional[int]                  = Field(default=500,  ge=1)
    stream:      Optional[bool]                 = Field(default=False)
    top_p:       Optional[float]                = Field(default=1.0,  ge=0.0, le=1.0)
    stop:        Optional[Union[str, List[str]]] = Field(default=None)


class EmbeddingRequest(BaseModel):
    model:           str                   = Field(default=DEFAULT_EMBEDDING_MODEL_ID)
    input:           Union[str, List[str]] = Field(..., description='Text or list of texts to embed')
    encoding_format: Optional[str]         = Field(default='float',
                                                   description="'float' or 'base64' (OpenAI-compatible)")
    dimensions:      Optional[int]         = Field(default=None,
                                                   description='Truncate output vectors to this size')
    user:            Optional[str]         = Field(default=None)
    # Extension: task instruction prefix used by Instructor models
    instruction:     Optional[str]         = Field(default=None,
                                                   description='Task instruction for hkunlp/instructor-* models')


class CombinedServer:
    def __init__(
        self,
        llm_model_name:      str = LLM_MODEL_REPO,
        llm_model_file:      str = LLM_MODEL_FILE,
        n_ctx:               int = 10048,
        n_gpu_layers:        int = -1,
        max_concurrent_requests: int = 3,
    ):
        logger.info(f'Downloading LLM: {llm_model_name}/{llm_model_file}')
        model_path = hf_hub_download(repo_id=llm_model_name, filename=llm_model_file)
        logger.info(f'LLM downloaded to: {model_path}')

        logger.info('Loading LLM into memory (GPU offload)...')
        self.llm = Llama(
            model_path=model_path,
            n_ctx=n_ctx,
            n_gpu_layers=n_gpu_layers,
            verbose=False
        )
        logger.info('LLM loaded successfully.')

        logger.info(f'Loading embedding model: {EMBEDDING_MODEL_E5_ID}')
        self.embedder_e5 = SentenceTransformer(EMBEDDING_MODEL_E5_ID)
        logger.info(f'E5 embedding model loaded — dim={EMBEDDING_MODEL_E5_DIM}')

        logger.info(f'Loading embedding model: {EMBEDDING_MODEL_INSTRUCTOR_ID}')
        self.embedder_instructor = INSTRUCTOR(EMBEDDING_MODEL_INSTRUCTOR_ID)
        logger.info(f'Instructor embedding model loaded — dim={EMBEDDING_MODEL_INSTRUCTOR_DIM}')

        self.semaphore      = asyncio.Semaphore(max_concurrent_requests)
        self.llm_lock       = threading.Lock()
        self.embed_e5_lock  = threading.Lock()
        self.embed_ins_lock = threading.Lock()

        self.app = FastAPI(
            title='GPT-OSS-20B + Multilingual-E5 + Instructor-Large OpenAI-Compatible API',
            version='2.0.0'
        )
        self._setup_routes()
        logger.info('Server initialised.')

    # ── helpers ────────────────────────────────────────────────────────────────

    @staticmethod
    def _resolve_embed_model(model_id: str) -> str:
        """Normalise a user-supplied model name to a canonical HF model ID."""
        key = (model_id or '').strip()
        resolved = MODEL_ALIASES.get(key)
        if resolved is None:
            raise HTTPException(
                status_code=400,
                detail=(
                    f"Unknown embedding model '{key}'. "
                    f"Supported: '{EMBEDDING_MODEL_E5_ID}', '{EMBEDDING_MODEL_INSTRUCTOR_ID}' "
                    f"(and their short aliases)."
                )
            )
        return resolved

    @staticmethod
    def _encode_vector(vec: np.ndarray, fmt: str, dims: Optional[int]) -> Any:
        """Apply optional dimension truncation and serialise to float list or base64."""
        if dims is not None:
            vec = vec[:dims]
        if fmt == 'base64':
            return base64.b64encode(vec.astype(np.float32).tobytes()).decode('utf-8')
        return vec.tolist()   # 'float' — default OpenAI format

    def _count_tokens(self, texts: List[str]) -> List[int]:
        """
        Count tokens per text using the LLM tokeniser (most accurate proxy
        available without loading a separate sentence-piece model).
        Falls back to whitespace-split word count if tokenisation raises.
        """
        counts = []
        for t in texts:
            try:
                counts.append(len(self.llm.tokenize(t.encode('utf-8'), add_bos=False)))
            except Exception:
                counts.append(len(t.split()))
        return counts

    # ── routes ─────────────────────────────────────────────────────────────────

    def _setup_routes(self):

        @self.app.get('/')
        async def root():
            return {
                'message': 'GPT-OSS-20B + Multilingual-E5 + Instructor-Large OpenAI-Compatible API',
                'endpoints': {
                    'chat':        '/v1/chat/completions',
                    'completions': '/v1/completions',
                    'embeddings':  '/v1/embeddings',
                    'models':      '/v1/models',
                    'health':      '/health'
                }
            }

        @self.app.get('/health')
        async def health_check():
            return {
                'status':                'healthy',
                'llm_model':             LLM_MODEL_ID,
                'embedding_model_e5':    EMBEDDING_MODEL_E5_ID,
                'embedding_dim_e5':      EMBEDDING_MODEL_E5_DIM,
                'embedding_model_ins':   EMBEDDING_MODEL_INSTRUCTOR_ID,
                'embedding_dim_ins':     EMBEDDING_MODEL_INSTRUCTOR_DIM,
                'backend':               'llama-cpp-python + sentence-transformers + InstructorEmbedding'
            }

        @self.app.get('/v1/models')
        async def list_models():
            ts = int(time.time())
            return {
                'object': 'list',
                'data': [
                    {'id': LLM_MODEL_ID,                 'object': 'model', 'created': ts, 'owned_by': 'organization', 'permission': [], 'root': LLM_MODEL_ID,                 'parent': None},
                    {'id': EMBEDDING_MODEL_E5_ID,         'object': 'model', 'created': ts, 'owned_by': 'organization', 'permission': [], 'root': EMBEDDING_MODEL_E5_ID,         'parent': None},
                    {'id': EMBEDDING_MODEL_INSTRUCTOR_ID, 'object': 'model', 'created': ts, 'owned_by': 'organization', 'permission': [], 'root': EMBEDDING_MODEL_INSTRUCTOR_ID, 'parent': None},
                ]
            }

        @self.app.post('/v1/chat/completions')
        async def create_chat_completion(request: ChatCompletionRequest):
            if request.stream:
                return StreamingResponse(
                    self._stream_chat_completion(request),
                    media_type='text/event-stream'
                )
            return await self._chat_completion_response(request)

        @self.app.post('/v1/completions')
        async def create_completion(request: CompletionRequest):
            if request.stream:
                return StreamingResponse(
                    self._stream_completion(request),
                    media_type='text/event-stream'
                )
            return await self._completion_response(request)

        @self.app.post('/v1/embeddings')
        async def create_embeddings(request: EmbeddingRequest):
            return await self._embedding_response(request)

    # ── chat / completion internals (unchanged) ────────────────────────────────

    def _messages_to_prompt(self, messages: List[Message]) -> str:
        parts = []
        for msg in messages:
            if msg.role == 'system':
                parts.append(f'<|system|>\n{msg.content}')
            elif msg.role == 'user':
                parts.append(f'<|user|>\n{msg.content}<|end|>')
            elif msg.role == 'assistant':
                parts.append(f'<|start|>assistant<|channel|>final<|message|>\n{msg.content}<|end|>')
        parts.append('<|start|>assistant<|channel|>final<|message|>\n')
        return '\n'.join(parts)

    def _stop_sequences(self, stop: Optional[Union[str, List[str]]]) -> List[str]:
        defaults = ['<|end|>', '<|user|>']
        if stop is None:
            return defaults
        if isinstance(stop, str):
            return defaults + [stop]
        return defaults + stop

    async def _chat_completion_response(self, request: ChatCompletionRequest) -> Dict:
        async with self.semaphore:
            try:
                prompt = self._messages_to_prompt(request.messages)
                stops  = self._stop_sequences(request.stop)
                with self.llm_lock:
                    resp = self.llm(prompt, max_tokens=request.max_tokens, temperature=request.temperature, top_p=request.top_p, stop=stops)
                text = resp['choices'][0]['text'].strip()
                return {
                    'id':      f'chatcmpl-{uuid.uuid4().hex[:8]}',
                    'object':  'chat.completion',
                    'created': int(time.time()),
                    'model':   request.model,
                    'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': text}, 'finish_reason': 'stop'}],
                    'usage':   {'prompt_tokens': len(prompt.split()), 'completion_tokens': len(text.split()), 'total_tokens': len(prompt.split()) + len(text.split())}
                }
            except Exception as e:
                logger.error(f'Chat completion error: {e}')
                raise HTTPException(status_code=500, detail=str(e))

    async def _stream_chat_completion(self, request: ChatCompletionRequest):
        async with self.semaphore:
            try:
                prompt  = self._messages_to_prompt(request.messages)
                stops   = self._stop_sequences(request.stop)
                cid     = f'chatcmpl-{uuid.uuid4().hex[:8]}'
                created = int(time.time())
                yield f"data: {json.dumps({'id': cid, 'object': 'chat.completion.chunk', 'created': created, 'model': request.model, 'choices': [{'index': 0, 'delta': {'role': 'assistant', 'content': ''}, 'finish_reason': None}]})}\n\n"
                with self.llm_lock:
                    for output in self.llm(prompt, max_tokens=request.max_tokens, temperature=request.temperature, top_p=request.top_p, stop=stops, stream=True):
                        token = output['choices'][0]['text']
                        yield f"data: {json.dumps({'id': cid, 'object': 'chat.completion.chunk', 'created': created, 'model': request.model, 'choices': [{'index': 0, 'delta': {'content': token}, 'finish_reason': None}]})}\n\n"
                yield f"data: {json.dumps({'id': cid, 'object': 'chat.completion.chunk', 'created': created, 'model': request.model, 'choices': [{'index': 0, 'delta': {}, 'finish_reason': 'stop'}]})}\n\n"
                yield 'data: [DONE]\n\n'
            except Exception as e:
                logger.error(f'Chat stream error: {e}')
                yield f"data: {json.dumps({'error': {'message': str(e), 'type': 'internal_error'}})}\n\n"

    async def _completion_response(self, request: CompletionRequest) -> Dict:
        async with self.semaphore:
            try:
                stops = self._stop_sequences(request.stop)
                with self.llm_lock:
                    resp = self.llm(request.prompt, max_tokens=request.max_tokens, temperature=request.temperature, top_p=request.top_p, stop=stops)
                text = resp['choices'][0]['text']
                return {
                    'id':      f'cmpl-{uuid.uuid4().hex[:8]}',
                    'object':  'text_completion',
                    'created': int(time.time()),
                    'model':   request.model,
                    'choices': [{'text': text, 'index': 0, 'finish_reason': 'stop'}],
                    'usage':   {'prompt_tokens': len(request.prompt.split()), 'completion_tokens': len(text.split()), 'total_tokens': len(request.prompt.split()) + len(text.split())}
                }
            except Exception as e:
                logger.error(f'Completion error: {e}')
                raise HTTPException(status_code=500, detail=str(e))

    async def _stream_completion(self, request: CompletionRequest):
        async with self.semaphore:
            try:
                stops   = self._stop_sequences(request.stop)
                cid     = f'cmpl-{uuid.uuid4().hex[:8]}'
                created = int(time.time())
                with self.llm_lock:
                    for output in self.llm(request.prompt, max_tokens=request.max_tokens, temperature=request.temperature, top_p=request.top_p, stop=stops, stream=True):
                        token = output['choices'][0]['text']
                        yield f"data: {json.dumps({'id': cid, 'object': 'text_completion', 'created': created, 'model': request.model, 'choices': [{'text': token, 'index': 0, 'finish_reason': None}]})}\n\n"
                yield f"data: {json.dumps({'id': cid, 'object': 'text_completion', 'created': created, 'model': request.model, 'choices': [{'text': '', 'index': 0, 'finish_reason': 'stop'}]})}\n\n"
                yield 'data: [DONE]\n\n'
            except Exception as e:
                logger.error(f'Completion stream error: {e}')
                yield f"data: {json.dumps({'error': {'message': str(e), 'type': 'internal_error'}})}\n\n"

    # ── embedding endpoint ─────────────────────────────────────────────────────

    async def _embedding_response(self, request: EmbeddingRequest) -> Dict:
        async with self.semaphore:
            try:
                # ── 1. Normalise input to List[str] ────────────────────────────
                texts: List[str] = (
                    [request.input] if isinstance(request.input, str) else list(request.input)
                )
                if not texts:
                    raise HTTPException(status_code=400, detail="'input' must not be empty.")

                # Guard: OpenAI caps batch size at 2048 items
                if len(texts) > 2048:
                    raise HTTPException(
                        status_code=400,
                        detail=f"Batch size {len(texts)} exceeds maximum of 2048."
                    )

                fmt  = (request.encoding_format or 'float').lower()
                dims = request.dimensions
                if fmt not in ('float', 'base64'):
                    raise HTTPException(
                        status_code=400,
                        detail=f"Unsupported encoding_format '{fmt}'. Use 'float' or 'base64'."
                    )

                # ── 2. Resolve model alias ─────────────────────────────────────
                used_model = self._resolve_embed_model(request.model)

                # ── 3. Encode ──────────────────────────────────────────────────
                if used_model == EMBEDDING_MODEL_INSTRUCTOR_ID:
                    raw_vecs = self._embed_instructor(texts, request.instruction)
                else:
                    raw_vecs = self._embed_e5(texts)

                # Validate dimensions argument
                max_dim = raw_vecs.shape[1]
                if dims is not None and not (1 <= dims <= max_dim):
                    raise HTTPException(
                        status_code=400,
                        detail=f"'dimensions' must be between 1 and {max_dim} for model '{used_model}'."
                    )

                # ── 4. Token counts (accurate, via LLM tokeniser) ──────────────
                token_counts  = self._count_tokens(texts)
                prompt_tokens = sum(token_counts)

                # ── 5. Build OpenAI-compatible response ────────────────────────
                data_list = [
                    {
                        'object':    'embedding',
                        'index':     idx,
                        'embedding': self._encode_vector(vec, fmt, dims),
                    }
                    for idx, vec in enumerate(raw_vecs)
                ]

                return {
                    'object': 'list',
                    'data':   data_list,
                    'model':  used_model,
                    'usage':  {
                        'prompt_tokens': prompt_tokens,
                        'total_tokens':  prompt_tokens,
                    },
                }

            except HTTPException:
                raise
            except Exception as e:
                logger.error(f'Embedding error: {e}')
                raise HTTPException(status_code=500, detail=str(e))

    # ── embedding model backends ───────────────────────────────────────────────

    def _embed_e5(self, texts: List[str]) -> np.ndarray:
        with self.embed_e5_lock:
            return self.embedder_e5.encode(
                texts,
                normalize_embeddings=True,
                show_progress_bar=False,
                batch_size=64,          # efficient for GPU batching
            )

    def _embed_instructor(self, texts: List[str], instruction: Optional[str] = None) -> np.ndarray:
        instr = instruction or 'Represent the sentence: '
        pairs = [[instr, t] for t in texts]
        with self.embed_ins_lock:
            vecs = self.embedder_instructor.encode(pairs, batch_size=32)
        norms = np.linalg.norm(vecs, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1, norms)
        return vecs / norms


# ── NgrokTunnelManager / ServerManager / entry-points (unchanged) ──────────────

class NgrokTunnelManager:

    def __init__(self):
        self.tunnel     = None
        self.public_url = None
        self.is_active  = False
        atexit.register(self.cleanup)

    def setup_auth(self, authtoken: str = None):
        try:
            token = authtoken or os.environ.get('NGROK_AUTHTOKEN')
            if token:
                ngrok.set_auth_token(token)
                logger.info('Ngrok auth token set.')
        except Exception as e:
            logger.warning(f'Ngrok auth warning: {e}')

    def create_tunnel(self, port: int = 8000):
        try:
            self.cleanup()
            logger.info(f'Creating ngrok tunnel on port {port}...')
            self.tunnel     = ngrok.connect(port, 'http')
            self.public_url = str(self.tunnel.public_url).rstrip('/')
            self.is_active  = True
            logger.info(f'Tunnel created: {self.public_url}')
            return self.public_url
        except Exception as e:
            logger.error(f'Tunnel creation failed: {e}')
            return None

    def test_tunnel(self, max_retries: int = 5) -> bool:
        if not self.public_url:
            return False
        for i in range(max_retries):
            try:
                r = requests.get(f'{self.public_url}/health', timeout=15)
                if r.status_code == 200:
                    logger.info('Tunnel health check passed.')
                    return True
            except Exception as e:
                if i == max_retries - 1:
                    logger.error(f'Tunnel test failed: {e}')
                else:
                    time.sleep(3)
        return False

    def cleanup(self):
        try:
            if self.tunnel:
                ngrok.disconnect(self.tunnel.public_url)
            ngrok.kill()
            self.tunnel     = None
            self.public_url = None
            self.is_active  = False
        except Exception as e:
            logger.warning(f'Cleanup warning: {e}')


class ServerManager:

    def __init__(self, server_instance: CombinedServer):
        self.server         = server_instance
        self.server_thread  = None
        self.tunnel_manager = NgrokTunnelManager()
        self.is_running     = False

    def start_with_ngrok(self, port: int = 8000, authtoken: str = None) -> bool:
        if self.is_running:
            print('Server is already running!')
            return True

        print(f'Starting Combined Server on port {port}...')
        try:
            self.tunnel_manager.setup_auth(authtoken)

            def _run():
                uvicorn.run(self.server.app, host='0.0.0.0', port=port, log_level='info')

            self.server_thread = threading.Thread(target=_run, daemon=True)
            self.server_thread.start()
            print('Waiting for server to start...')
            time.sleep(5)

            public_url = self.tunnel_manager.create_tunnel(port=port)
            if not public_url:
                print('Failed to create ngrok tunnel.')
                return False

            self.is_running = True
            time.sleep(3)

            if self.tunnel_manager.test_tunnel():
                print(f"\n{'='*65}")
                print('  GPT-OSS-20B + Multilingual-E5 + Instructor-Large Server Active!')
                print(f"{'='*65}")
                print(f'\n  Public URL : {public_url}')
                print(f'\n  Endpoints:')
                print(f'    POST  {public_url}/v1/chat/completions')
                print(f'    POST  {public_url}/v1/completions')
                print(f'    POST  {public_url}/v1/embeddings')
                print(f'    GET   {public_url}/v1/models')
                print(f'    GET   {public_url}/health')
                print(f'\n  Models:')
                print(f'    LLM            : {LLM_MODEL_ID}')
                print(f'    Embeddings (1) : {EMBEDDING_MODEL_E5_ID} (dim={EMBEDDING_MODEL_E5_DIM})')
                print(f'    Embeddings (2) : {EMBEDDING_MODEL_INSTRUCTOR_ID} (dim={EMBEDDING_MODEL_INSTRUCTOR_DIM})')
                print(f"{'='*65}\n")
                return True
            else:
                print('Tunnel test failed.')
                return False

        except Exception as e:
            print(f'Server start failed: {e}')
            return False

    def keep_alive_blocking(self, max_hours: int = 12):
        start_time = datetime.now()
        end_time   = start_time + timedelta(hours=max_hours)
        iteration  = 0

        print(f"\n{'='*65}")
        print('🟢  Keep-Alive Loop Active  (Cell is Running)')
        print(f"  Started  : {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"  Ends at  : {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f'  Max run  : {max_hours} hours')
        print(f'  URL      : {self.get_public_url()}')
        print(f'  Stop     : Press Interrupt / Ctrl+C')
        print(f"{'='*65}\n")

        try:
            while datetime.now() < end_time:
                iteration += 1
                current   = datetime.now()
                elapsed   = current - start_time
                remaining = end_time - current

                if iteration % 10 == 0:
                    h, rem   = divmod(int(elapsed.total_seconds()),   3600)
                    m, s     = divmod(rem, 60)
                    rh, rrem = divmod(int(remaining.total_seconds()), 3600)
                    rm, _    = divmod(rrem, 60)
                    print(f"[{current.strftime('%H:%M:%S')}] 🟢 Status  |  Elapsed: {h}h {m}m {s}s  |  Remaining: {rh}h {rm}m  |  Iter: {iteration}")
                    if self.is_running and self.get_public_url():
                        try:
                            r = requests.get(f'{self.get_public_url()}/health', timeout=10)
                            status = '✓ OK' if r.status_code == 200 else f'✗ {r.status_code}'
                        except Exception as e:
                            status = f'✗ {str(e)[:40]}'
                        print(f'                  Health  : {status}')
                    print()

                time.sleep(30)

        except KeyboardInterrupt:
            print('\n⚠️  Keep-alive interrupted by user.')
        except Exception as e:
            print(f'\n❌ Keep-alive error: {e}')
        finally:
            elapsed = datetime.now() - start_time
            h, rem  = divmod(int(elapsed.total_seconds()), 3600)
            m, s    = divmod(rem, 60)
            print(f"\n{'='*65}")
            print(f'🔴 Keep-Alive Ended  |  Total runtime: {h}h {m}m {s}s')
            print(f"{'='*65}\n")

    def stop(self):
        print('Stopping server tunnel...')
        self.tunnel_manager.cleanup()
        self.is_running = False
        print('Tunnel stopped.')

    def get_public_url(self) -> Optional[str]:
        return self.tunnel_manager.public_url


def start_server_and_keep_alive(
    llm_model_name: str = LLM_MODEL_REPO,
    llm_model_file: str = LLM_MODEL_FILE,
    n_ctx:          int = 10048,
    n_gpu_layers:   int = -1,
    max_requests:   int = 3,
    port:           int = 8000,
    authtoken:      str = None,
    max_hours:      int = 12
):
    print('=' * 65)
    print('  Initialising GPT-OSS-20B + E5 + Instructor-Large Combined Server')
    print('=' * 65)
    try:
        print('\n[1/2] Downloading & loading models (may take a few minutes)...')
        server = CombinedServer(
            llm_model_name=llm_model_name,
            llm_model_file=llm_model_file,
            n_ctx=n_ctx,
            n_gpu_layers=n_gpu_layers,
            max_concurrent_requests=max_requests
        )
        print('\n[2/2] Starting FastAPI server and ngrok tunnel...')
        manager = ServerManager(server)
        success = manager.start_with_ngrok(port=port, authtoken=authtoken)
        if success:
            print('\n⚠️  IMPORTANT: Cell will now BLOCK — [*] stays active to keep Kaggle alive.')
            print('   Press Interrupt / Ctrl+C to stop.\n')
            time.sleep(2)
            manager.keep_alive_blocking(max_hours=max_hours)
        else:
            print('❌ Server failed to start.')
    except Exception as e:
        print(f'❌ Fatal error: {e}')
        import traceback
        traceback.print_exc()


def start_server_only(
    llm_model_name: str = LLM_MODEL_REPO,
    llm_model_file: str = LLM_MODEL_FILE,
    n_ctx:          int = 10048,
    n_gpu_layers:   int = -1,
    max_requests:   int = 3,
    port:           int = 8000,
    authtoken:      str = None
):
    print('Initialising server (no keep-alive)...')
    try:
        server  = CombinedServer(
            llm_model_name=llm_model_name,
            llm_model_file=llm_model_file,
            n_ctx=n_ctx,
            n_gpu_layers=n_gpu_layers,
            max_concurrent_requests=max_requests
        )
        manager = ServerManager(server)
        success = manager.start_with_ngrok(port=port, authtoken=authtoken)
        if success:
            print('\n⚠️  WARNING: Keep-alive NOT active. Session may time out.')
            print('   Run: manager.keep_alive_blocking() to enable it.')
            return manager, server
        return None, None
    except Exception as e:
        print(f'❌ Fatal error: {e}')
        import traceback
        traceback.print_exc()
        return None, None


print('✅ Combined Server (GPT-OSS-20B + Multilingual-E5 + Instructor-Large) code loaded.')
print('\n⚡ Usage:')
print('  Option 1 (Recommended — keeps Kaggle session alive):')
print("    start_server_and_keep_alive()")
print('\n  Option 2 (Testing — returns immediately):')
print("    manager, server = start_server_only()")

## Cell 4 — Launch Server

In [ ]:
start_server_and_keep_alive(
    port=8001,
    max_hours=12
    # authtoken is read automatically from os.environ['NGROK_AUTHTOKEN'] set in Cell 2
)

## Cell 5 — Quick API Test

In [ ]:
import requests, json

PUBLIC_URL = 'PASTE_YOUR_NGROK_URL_HERE'.rstrip('/')

# ── Health ──
print('── Health ──')
print(requests.get(f'{PUBLIC_URL}/health').json())

# ── Models ──
print('\n── Models ──')
print(requests.get(f'{PUBLIC_URL}/v1/models').json())

# ── Chat Completion ──
print('\n── Chat Completion ──')
chat_resp = requests.post(
    f'{PUBLIC_URL}/v1/chat/completions',
    json={
        'model': 'gpt-oss-20b',
        'messages': [
            {'role': 'system',  'content': 'You are a helpful assistant. Be concise.'},
            {'role': 'user',    'content': 'What is machine learning in one sentence?'}
        ],
        'max_tokens': 100,
        'temperature': 0.7
    },
    timeout=60
)
print(chat_resp.json()['choices'][0]['message']['content'])

# ── Embeddings — single string ──
print('\n── Embeddings: single string ──')
single_resp = requests.post(
    f'{PUBLIC_URL}/v1/embeddings',
    json={
        'model': 'intfloat/multilingual-e5-large',
        'input': 'query: What is artificial intelligence?'
    },
    timeout=30
)
result = single_resp.json()
print(f"Embedding dim  : {len(result['data'][0]['embedding'])}")
print(f"Usage tokens   : {result['usage']}")

# ── Embeddings — batch (List[str]) ──
print('\n── Embeddings: batch ──')
batch_resp = requests.post(
    f'{PUBLIC_URL}/v1/embeddings',
    json={
        'model': 'intfloat/multilingual-e5-large',
        'input': [
            'passage: Artificial intelligence is the simulation of human intelligence.',
            'passage: মেশিন লার্নিং একটি কৃত্রিম বুদ্ধিমত্তার শাখা।',
            "passage: L'apprentissage automatique est une branche de l'IA."
        ]
    },
    timeout=30
)
batch_result = batch_resp.json()
for item in batch_result['data']:
    print(f"  index {item['index']}  ->  dim={len(item['embedding'])}")
print(f"Total usage: {batch_result['usage']}")

# ── Embeddings — base64 encoding format ──
print('\n── Embeddings: base64 encoding_format ──')
b64_resp = requests.post(
    f'{PUBLIC_URL}/v1/embeddings',
    json={
        'model': 'intfloat/multilingual-e5-large',
        'input': 'Hello world',
        'encoding_format': 'base64'
    },
    timeout=30
)
b64_result = b64_resp.json()
print(f"encoding_format : base64")
print(f"embedding type  : {type(b64_result['data'][0]['embedding'])}")
print(f"base64 length   : {len(b64_result['data'][0]['embedding'])} chars")

# ── Embeddings — dimensions truncation ──
print('\n── Embeddings: dimensions truncation ──')
dim_resp = requests.post(
    f'{PUBLIC_URL}/v1/embeddings',
    json={
        'model': 'intfloat/multilingual-e5-large',
        'input': ['First sentence.', 'Second sentence.'],
        'dimensions': 256
    },
    timeout=30
)
dim_result = dim_resp.json()
for item in dim_result['data']:
    print(f"  index {item['index']}  ->  dim={len(item['embedding'])}  (truncated to 256)")

# ── Embeddings — instructor-large, default instruction ──
print('\n── Embeddings: hkunlp/instructor-large (default instruction) ──')
ins_resp = requests.post(
    f'{PUBLIC_URL}/v1/embeddings',
    json={
        'model': 'hkunlp/instructor-large',
        'input': 'What is artificial intelligence?'
    },
    timeout=30
)
ins_result = ins_resp.json()
print(f"Embedding dim  : {len(ins_result['data'][0]['embedding'])}")
print(f"Usage tokens   : {ins_result['usage']}")

# ── Embeddings — instructor-large, batch + custom instruction ──
print('\n── Embeddings: hkunlp/instructor-large (batch, custom instruction) ──')
ins_batch_resp = requests.post(
    f'{PUBLIC_URL}/v1/embeddings',
    json={
        'model': 'hkunlp/instructor-large',
        'input': [
            'Artificial intelligence is the simulation of human intelligence.',
            'Machine learning is a branch of AI.',
            'Deep learning uses neural networks.'
        ],
        'instruction': 'Represent the document for retrieval: '
    },
    timeout=30
)
ins_batch_result = ins_batch_resp.json()
for item in ins_batch_result['data']:
    print(f"  index {item['index']}  ->  dim={len(item['embedding'])}")
print(f"Total usage: {ins_batch_result['usage']}")

# ── Embeddings — short model alias ──
print('\n── Embeddings: short alias (e5-large) ──')
alias_resp = requests.post(
    f'{PUBLIC_URL}/v1/embeddings',
    json={
        'model': 'e5-large',
        'input': ['alias resolution test']
    },
    timeout=30
)
alias_result = alias_resp.json()
print(f"Resolved model : {alias_result['model']}")
print(f"Embedding dim  : {len(alias_result['data'][0]['embedding'])}")